# ezTrack — Location Tracking (batch)

Apply one set of selections and tracking parameters to every video in a folder. Draw the crop / mask / ROIs / scale once on the first video (this assumes every video shares the same framing); a fresh reference frame is built per video.

## 1. Load packages

In [ ]:
import holoviews as hv
import eztrack as ez

hv.extension("bokeh")

## 2. Point at the folder
`ftype` is the video file extension to process. `region_names` names the ROIs to draw.

In [ ]:
session = ez.Session(
    dpath="../../PracticeVideos/",
    ftype="mp4",
    start=0,
    end=None,
    # Speed knobs (reduction factors, 1 = off); outputs stay in original space.
    spatial_downsample=1,  # 2 = track at half resolution
    temporal_downsample=1,  # 2 = track every other frame (one row per tracked frame)
    region_names=["left", "right"],
)

ez.discover_files(session)
session.file_names

## 3. Draw selections on the first video
The crop/mask/ROIs/scale you draw here are applied to every file. We load the first file and build its reference so the ROI tool has an image to draw on.

In [ ]:
session.file = session.file_names[0]

hv.output(size=50)
ez.crop_tool(session)

In [ ]:
hv.output(size=100)
ez.mask_tool(session)

In [ ]:
ez.reference_frame(session, num_frames=50)

hv.output(size=100)
ez.roi_tool(session)

## 4. (Optional) Scale

In [ ]:
hv.output(size=100)
ez.distance_tool(session)

In [ ]:
ez.set_scale(session, real_distance=100, unit="cm")

## 5. (Optional) Save / reuse the selections
`session.selections.save("selections.json")` — or load a previously saved set with `session.selections = ez.Selections.load("selections.json")`.

In [ ]:
session.selections.save("selections.json")

## 6. Set tracking parameters
Same options as the single-video notebook (step 9 there explains each): `threshold_pct` (or `threshold_abs` for a fixed 0–255 cutoff instead of the percentile, with `threshold_on` choosing whether that cutoff is measured against the baseline difference or the raw pixel value), `method`, `window`, and `denoise`/`denoise_kernel` (drop specks / a thin wire). These apply to every video in the batch. (Position outlier removal is applied in the run step below, since the batch has no interactive per-file review.)

In [ ]:
params = ez.TrackParams(
    threshold_pct=99.5,  # used only when threshold_abs is None
    threshold_abs=None,  # fixed cutoff (0-255); overrides the percentile when set
    threshold_on="difference",  # what threshold_abs measures: "difference" (vs baseline) | "raw" (pixel value)
    method="abs",  # "abs" | "light" | "dark"  (raw mode needs "dark" or "light")
    window=ez.Window(size=100, weight=0.9),  # or window=None to disable
    denoise=False,  # True = remove small specks / thin wire from the mask
    denoise_kernel=5,  # opening kernel in px (only used when denoise=True)
)

## 7. Run the batch
Writes `<video>_tracking.csv` per file and a combined `BatchSummary.csv`, and returns the summary table plus a trace + heatmap for each video. Optionally pass `bins={...}` to summarize per time bin, and `hampel=(window, sigma)` to remove position outliers from every track before summarizing (e.g. `hampel=(7, 3.0)`).

In [ ]:
summary, panels = ez.batch_process(session, params, bins=None, hampel=None)
summary

In [ ]:
hv.output(size=80)
panels.cols(2)